# Aula 9 — Redes neurais convolucionais

**Processamento de Linguagem Natural e Imagens · 2026.2 · Prof. James Moraes de Almeida**

**O enigma.** As fotos do CIFAR-10 têm 32 × 32 px. Em cada uma, os 1.024 pixels mudam
de lugar segundo uma permutação fixa, a mesma em todas as fotos: para os olhos, cada
foto vira um mosaico sem objeto. Duas redes são treinadas do zero, uma
densa (MLP) e uma convolucional (CNN), nas fotos e nas embaralhadas. Qual delas perde
com o embaralhamento?

Este notebook refaz as medidas da aula com as mesmas funções que produziram os números
dos slides:

0. o ambiente e o tempo de treino desta máquina;
1. os dados, carregados uma única vez;
2. o enigma montado: embaralhar e desembaralhar;
3. a MLP nas fotos e nas embaralhadas;
4. a identidade que explica a MLP;
5. vizinhos se parecem: a aposta da CNN;
6. uma camada convolucional à mão;
7. a CNN da aula;
8. dentro da CNN: kernels, mapas e campo receptivo;
9. onde a rede erra;
10. a CNN nas embaralhadas;
11. sobreajuste, aumento de dados e *callbacks*;
12. exercícios;
13. apêndice opcional: uma rede pré-treinada.

## Como usar este notebook

- **Modo.** A primeira célula de código define `MODO`. No modo `"rapido"`, cada rede
  treina com 10.000 fotos por 5 épocas; no modo `"completo"`, com
  45.000 fotos por 10 épocas, o protocolo dos slides. Depois de
  trocar o modo, o notebook tem de rodar de novo desde o início.
- **Tempo.** Toda célula que treina imprime quanto levou, e a seção 0 estima o tempo de
  cada treino nesta máquina antes de começar.
- **Números.** Os números do professor entram numa célula própria, copiados dos
  arquivos da análise. Hardware, número de *threads* e versão das bibliotecas mudam os
  últimos dígitos, e cada treino sorteia pesos iniciais diferentes. A comparação certa
  é com a faixa de ± 2σ das sementes do professor, no mesmo modo.
- **Saídas.** Modelos e tabelas vão para a pasta `saida/`, criada pelo próprio notebook.

# 0 — Ambiente

O notebook usa o Keras 3 com o TensorFlow 2.21, em CPU; não precisa de GPU. No ambiente
da disciplina:

```bash
conda activate img_proc
pip install tensorflow==2.21.0
```

O `pip` instala o Keras 3 junto. No Google Colab, os dois já vêm instalados; a versão
pode ser outra, e os últimos dígitos dos resultados podem mudar.

In [ ]:
MODO = "rapido"     # "rapido" ou "completo"
# rápido: 10.000 fotos de treino, 5 épocas · completo: 45.000 fotos, 10 épocas (os slides)

assert MODO in ("rapido", "completo"), MODO
EPOCAS = {"rapido": 5, "completo": 10}[MODO]
EPOCAS_SOBREAJUSTE = {"rapido": 15, "completo": 40}[MODO]      # seção 11
LOTE = 64                                                     # fotos por passo do otimizador

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")      # menos avisos do TensorFlow

import hashlib
import inspect
import json
import shutil
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
import tensorflow as tf

print("Keras", keras.__version__, "· TensorFlow", tf.__version__, "· NumPy", np.__version__)
print("núcleos de CPU:", os.cpu_count())
assert keras.__version__.startswith("3."), "este notebook usa o Keras 3"

keras.utils.set_random_seed(0)     # sorteios reprodutíveis; os últimos dígitos ainda variam
SAIDA = Path("saida")
SAIDA.mkdir(exist_ok=True)
plt.rcParams["figure.dpi"] = 110


def br(x, casas=1):
    """Número no formato do texto: vírgula decimal e ponto de milhar."""
    return f"{x:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def pct(x, casas=1):
    """Fração de 0 a 1 escrita em porcentagem."""
    return br(100 * x, casas) + " %"


@contextmanager
def cronometro(rotulo):
    """Imprime quanto tempo o bloco levou."""
    t0 = time.perf_counter()
    yield
    print(f"tempo de {rotulo}: {br(time.perf_counter() - t0, 0)} s")

## O protocolo da aula

A célula abaixo é uma cópia exata de `protocolo.py`, o arquivo que define os dados e as
duas redes de todos os treinos dos slides. O notebook não reescreve nenhuma
arquitetura: uma segunda versão seria o lugar onde o número do slide se descola do
número do notebook. As funções importam o Keras por dentro.

In [ ]:
# O protocolo da aula: os dados, o embaralhamento e as duas redes.
# Cópia exata de animacoes/treino/protocolo.py (sha256 b47b52e29002), feita pelo montar_notebook.py.
# Não altere esta célula: uma segunda versão da arquitetura é onde o número do slide
# se descola do número do notebook.

SEMENTE_PERM = 2026
LADO = 32
CLASSES = ["avião", "automóvel", "pássaro", "gato", "cervo",
           "cachorro", "sapo", "cavalo", "navio", "caminhão"]
N_TREINO = 45_000
N_POR_CLASSE_PEQUENO = 1_000     # E4 e modo rápido: 10.000 fotos de treino


def carregar_cifar10():
    """(x_tr, y_tr), (x_val, y_val), (x_te, y_te), em float32 de 0 a 1 e rótulos 1D."""
    import keras
    (x, y), (x_te, y_te) = keras.datasets.cifar10.load_data()
    x = x.astype("float32") / 255.0
    x_te = x_te.astype("float32") / 255.0
    y, y_te = y.ravel(), y_te.ravel()
    return ((x[:N_TREINO], y[:N_TREINO]), (x[N_TREINO:], y[N_TREINO:]), (x_te, y_te))


def subconjunto_pequeno(y_tr: np.ndarray, n_por_classe: int = N_POR_CLASSE_PEQUENO,
                        semente: int = SEMENTE_PERM) -> np.ndarray:
    """Índices de n fotos por classe, sorteados dentro das 45.000 de treino."""
    rng = np.random.default_rng(semente)
    idx = [rng.choice(np.flatnonzero(y_tr == c), n_por_classe, replace=False)
           for c in range(10)]
    return np.sort(np.concatenate(idx))


def permutacao(semente: int = SEMENTE_PERM) -> np.ndarray:
    """A permutação das 1.024 posições: o pixel da posição q da foto embaralhada é o
    pixel perm[q] da foto original."""
    return np.random.default_rng(semente).permutation(LADO * LADO)


def permutacao_blocos(b: int, semente: int = SEMENTE_PERM) -> np.ndarray:
    """Permutação que só troca pixels DENTRO de cada bloco b × b (b divide 32).

    Um sorteio independente por bloco, na ordem das linhas. Com b = 32 há um bloco só,
    e o sorteio é exatamente `permutacao()`: a mesma chamada ao mesmo gerador.
    """
    assert LADO % b == 0
    rng = np.random.default_rng(semente)
    perm = np.empty(LADO * LADO, int)
    for bi in range(0, LADO, b):
        for bj in range(0, LADO, b):
            pos = np.array([(bi + u) * LADO + bj + v for u in range(b) for v in range(b)])
            perm[pos] = pos[rng.permutation(b * b)]
    return perm


def embaralhar(x: np.ndarray, perm: np.ndarray) -> np.ndarray:
    """Leva cada pixel (com os 3 canais juntos) para a posição da permutação."""
    n = x.shape[0]
    return x.reshape(n, LADO * LADO, 3)[:, perm].reshape(n, LADO, LADO, 3)


def construir_mlp():
    """A rede densa: 3.072 números em fila → 512 → 256 → 10. 1.707.274 parâmetros."""
    import keras
    from keras import layers
    return keras.Sequential([
        keras.Input((LADO, LADO, 3)),
        layers.Flatten(name="fila"),
        layers.Dense(512, activation="relu", name="densa1"),
        layers.Dense(256, activation="relu", name="densa2"),
        layers.Dense(10, activation="softmax", name="saida"),
    ], name="mlp")


def construir_cnn(aumento: bool = False):
    """A CNN da aula: 3 × [Conv 3×3 → ReLU → MaxPool 2×2], depois Flatten → 128 → 10.

    A ReLU é uma camada própria, e não `activation=` da Conv2D, para dar para ler o
    mapa antes e depois dela. 356.810 parâmetros.

    `aumento=True` põe na frente o aumento de dados como camadas: espelho horizontal
    e deslocamento de até 10 % (3,2 px) em cada eixo, com a borda refletida. Elas só
    agem no treino (fit); em evaluate e predict deixam a foto passar intacta. Não
    acrescentam parâmetros.
    """
    import keras
    from keras import layers
    camadas = [keras.Input((LADO, LADO, 3))]
    if aumento:
        camadas += [layers.RandomFlip("horizontal", name="espelho"),
                    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect", name="desloca")]
    for k, n in enumerate((32, 64, 128), start=1):
        camadas += [layers.Conv2D(n, 3, padding="same", name=f"conv{k}"),
                    layers.ReLU(name=f"relu{k}"),
                    layers.MaxPooling2D(2, name=f"pool{k}")]
    camadas += [layers.Flatten(name="fila"),
                layers.Dense(128, activation="relu", name="densa"),
                layers.Dense(10, activation="softmax", name="saida")]
    return keras.Sequential(camadas, name="cnn")


def compilar(modelo):
    import keras
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo


PARAMS_MLP = 3072 * 512 + 512 + 512 * 256 + 256 + 256 * 10 + 10           # 1.707.274
PARAMS_CNN_CAMADAS = {
    "conv1": 3 * 3 * 3 * 32 + 32,        # 896
    "conv2": 3 * 3 * 32 * 64 + 64,       # 18.496
    "conv3": 3 * 3 * 64 * 128 + 128,     # 73.856
    "densa": 4 * 4 * 128 * 128 + 128,    # 262.272
    "saida": 128 * 10 + 10,              # 1.290
}
PARAMS_CNN = sum(PARAMS_CNN_CAMADAS.values())                               # 356.810
assert PARAMS_MLP == 1_707_274 and PARAMS_CNN == 356_810

## Os números do professor

Os treinos dos slides foram repetidos com várias **sementes**: a semente fixa o sorteio
dos pesos iniciais e da ordem dos lotes. A célula abaixo traz a acurácia no teste de
cada semente, copiada dos arquivos da análise pelo gerador deste notebook; nenhum número
foi digitado à mão. O desvio-padrão σ entre as sementes mede o **ruído de semente**: um
resultado a menos de 2σ da média do professor, no mesmo modo, é compatível com ele.
Com poucas sementes, o próprio σ é uma estimativa grosseira.

In [ ]:
# Os números do professor, copiados pelo montar_notebook.py dos arquivos da análise.
# Não altere à mão. Acurácias no teste (fração de 0 a 1), uma por semente.
# Fonte: animacoes/treino/saida/analise.json de 06/11/2026 00:32 (final, 5 sementes por configuração);
#        e10_rapido.json de 06/11/2026 00:31.
REFERENCIA = {'completo': {'fonte': 'E1, 45.000 fotos, 10 épocas, 5 sementes',
              'mlp_fotos': [0.4885, 0.4553, 0.4609, 0.4556, 0.467],
              'mlp_emb': [0.4732, 0.4647, 0.4546, 0.4654, 0.4516],
              'cnn_fotos': [0.7118, 0.6977, 0.6756, 0.6786, 0.7024],
              'cnn_emb': [0.4623, 0.4768, 0.4416, 0.4698, 0.4712]},
 'rapido': {'fonte': 'E10, 10.000 fotos, 5 épocas, 3 sementes',
            'mlp_fotos': [0.3746, 0.3764, 0.391],
            'mlp_emb': [0.3777, 0.3822, 0.3841],
            'cnn_fotos': [0.5413, 0.5805, 0.5732],
            'cnn_emb': [0.3942, 0.3934, 0.4012]}}

REF_ANALISE = {'heroi': 1,
 'campo_receptivo_px': {'conv1': 3,
                        'pool1': 4,
                        'conv2': 8,
                        'pool2': 10,
                        'conv3': 18,
                        'pool3': 22},
 'correlacao': {'fotos_h': [0.928663, 0.822329, 0.745014, 0.681724, 0.628408, 0.582875,
                            0.543533, 0.509633, 0.480495, 0.456187, 0.436249, 0.420349,
                            0.407965, 0.398945, 0.392969, 0.389965],
                'emb_h': [0.264039, 0.260001, 0.254477, 0.255813, 0.266967, 0.259338,
                          0.268707, 0.270818, 0.268687, 0.258521, 0.262084, 0.264259,
                          0.274203, 0.264437, 0.280041, 0.258712],
                'fotos_h_centrada': [0.903805, 0.761763, 0.659869, 0.577377, 0.50852,
                                     0.450098, 0.399735, 0.356125, 0.318222, 0.285838,
                                     0.25825, 0.23485, 0.214884, 0.198073, 0.184074,
                                     0.172758],
                'emb_h_centrada': [0.002972, -0.003392, -0.010159, -0.007378, 0.006874,
                                   -0.00169, 0.009844, 0.011694, 0.007457, -0.007631,
                                   -0.003253, -0.001622, 0.012224, -0.000922, 0.019788,
                                   -0.005256]},
 'distancias': {'vizinhos_reais_px': 1.203819,
                'vizinhos_embaralhados_px': 16.730699,
                'acaso_px': 16.693305},
 'rotas': {'A1': 0.46546,
           'sigma1': 0.013734,
           'A2': 0.4619,
           'sigma2': 0.008763,
           'dif': 0.00356,
           'limite': 0.014571,
           'passa': True},
 'rota1_iguais_float32': [10000, 10000, 10000, 10000, 10000],
 'confusao': {'par': [9, 1],
              'contagem': 192,
              'acc_por_classe': [0.835, 0.914, 0.659, 0.407, 0.647, 0.612, 0.819, 0.751,
                                 0.806, 0.668]},
 'e4': {'sem': {'treino_acc': [0.0984, 0.4304, 0.5087, 0.5598, 0.6064, 0.6451, 0.6719,
                               0.7091, 0.7321, 0.7577, 0.7746, 0.7902, 0.8063, 0.8123,
                               0.8514, 0.8549, 0.8214, 0.829, 0.8728, 0.8683, 0.8715, 0.888,
                               0.8769, 0.9, 0.9207, 0.9226, 0.9322, 0.9431, 0.9453, 0.9418,
                               0.9531, 0.9513, 0.9613, 0.9653, 0.9683, 0.9627, 0.9659,
                               0.9725, 0.9642, 0.9773, 0.9624],
                'val_acc': [0.0998, 0.4147, 0.4789, 0.5194, 0.5501, 0.5721, 0.5825, 0.5978,
                            0.6061, 0.6099, 0.6089, 0.6074, 0.6113, 0.6009, 0.6137, 0.6091,
                            0.5829, 0.5813, 0.5981, 0.5921, 0.5845, 0.5956, 0.5843, 0.5909,
                            0.5943, 0.5941, 0.5907, 0.5987, 0.5915, 0.5871, 0.5988, 0.5956,
                            0.5976, 0.6036, 0.6049, 0.6017, 0.5975, 0.6046, 0.6036, 0.6013,
                            0.5955],
                'val_perda': [2.3038, 1.5831, 1.4308, 1.3398, 1.2654, 1.217, 1.209, 1.172,
                              1.1759, 1.1995, 1.2573, 1.3154, 1.3715, 1.4641, 1.4295,
                              1.5273, 1.7604, 1.8666, 1.8577, 2.0458, 2.1368, 2.1671,
                              2.3681, 2.3631, 2.3218, 2.4875, 2.5506, 2.5834, 2.5778,
                              2.7334, 2.7358, 2.8805, 2.7849, 2.8538, 2.8045, 2.9947,
                              3.0713, 3.1092, 3.2044, 3.2007, 3.3862],
                'teste_acc': [0.5888, 0.5816, 0.5844],
                'early_stopping': [{'parada': 12, 'melhor': 7}, {'parada': 13, 'melhor': 8},
                                   {'parada': 12, 'melhor': 7}]},
        'com': {'treino_acc': [0.1011, 0.4035, 0.4696, 0.5098, 0.5623, 0.5949, 0.6228,
                               0.6425, 0.6496, 0.6784, 0.6791, 0.6919, 0.7059, 0.7064,
                               0.7204, 0.7326, 0.7313, 0.7446, 0.7439, 0.7607, 0.763,
                               0.7648, 0.7689, 0.7744, 0.7874, 0.7786, 0.7835, 0.7819,
                               0.7771, 0.8026, 0.8133, 0.8059, 0.8106, 0.8202, 0.827, 0.827,
                               0.8314, 0.8215, 0.823, 0.8302, 0.8424],
                'val_acc': [0.1021, 0.3945, 0.4568, 0.4966, 0.5391, 0.5748, 0.5956, 0.6166,
                            0.6172, 0.639, 0.6359, 0.6495, 0.6615, 0.6573, 0.6693, 0.675,
                            0.6777, 0.6841, 0.679, 0.6875, 0.6843, 0.6869, 0.689, 0.6885,
                            0.6989, 0.6874, 0.6897, 0.6895, 0.6824, 0.6989, 0.7039, 0.6933,
                            0.6986, 0.7026, 0.6991, 0.6981, 0.7013, 0.6923, 0.6877, 0.6922,
                            0.7045],
                'val_perda': [2.3045, 1.6434, 1.5012, 1.3907, 1.279, 1.1855, 1.1326, 1.0801,
                              1.0868, 1.025, 1.034, 1.0202, 0.9832, 0.9982, 0.9709, 0.9556,
                              0.9587, 0.9456, 0.9778, 0.9362, 0.9565, 0.9586, 0.9626, 0.964,
                              0.9467, 0.9678, 0.9911, 0.9874, 1.0187, 0.9639, 0.9632,
                              0.9822, 0.9891, 0.9759, 0.9674, 0.9921, 0.9827, 1.036, 1.0655,
                              1.0533, 1.0136],
                'teste_acc': [0.6803, 0.7019, 0.7004],
                'early_stopping': [{'parada': 17, 'melhor': 12},
                                   {'parada': 24, 'melhor': 19},
                                   {'parada': 27, 'melhor': 22}]},
        'sementes': 3,
        'epocas': 40}}

HEROI = REF_ANALISE["heroi"]        # o navio da aula, pela regra escrita na análise

# Sem o E10 na geração: procura o arquivo do professor ao lado do notebook.
_e10 = Path("e10_rapido.json")
if REFERENCIA["rapido"] is None and _e10.exists():
    _d = json.loads(_e10.read_text(encoding="utf-8"))
    REFERENCIA["rapido"] = {"fonte": "E10, lido de e10_rapido.json"}
    for _k, _v in _d.items():
        REFERENCIA["rapido"][_k] = [r["teste_acc"] for r in sorted(_v, key=lambda r: r["semente"])]
for _modo, _ref in REFERENCIA.items():
    print(f"modo {_modo}: {_ref['fonte'] if _ref else 'a medir'}")

In [ ]:
NOMES = {"mlp_fotos": "MLP, fotos", "mlp_emb": "MLP, embaralhadas",
         "cnn_fotos": "CNN, fotos", "cnn_emb": "CNN, embaralhadas"}
ACC = {}        # acurácia no teste de cada rede treinada neste notebook


def sigma_do_modo(ref):
    """O desvio entre sementes, juntando as 4 configurações do modo.

    Com 3 sementes, o desvio de UMA configuração é grosseiro: pode sair 0,3 ponto
    percentual por acaso e reprovar um treino normal. As 4 configurações têm o mesmo
    tipo de ruído, então a média das 4 variâncias estima melhor o ruído de um treino.
    """
    variancias = [np.var(ref[k], ddof=1) for k in NOMES if k in ref and len(ref[k]) >= 2]
    return float(np.sqrt(np.mean(variancias))) if variancias else None


def comparar(chaves):
    """As acurácias deste notebook ao lado das do professor no mesmo modo."""
    ref = REFERENCIA[MODO]
    print(f"modo {MODO} · professor: {ref['fonte'] if ref else 'a medir'}")
    s = sigma_do_modo(ref) if ref else None
    if s is not None:
        print(f"σ: desvio entre sementes, juntando as 4 configurações: {br(100 * s)} pontos percentuais")
    print(f"{'':20s}{'aqui':>9s}{'professor (média ± σ)':>25s}")
    for k in chaves:
        linha = f"{NOMES[k]:20s}{pct(ACC[k]):>9s}"
        v = np.asarray(ref[k]) if ref else np.array([])
        if len(v) >= 2 and s is not None:
            m = v.mean()
            # um treino seu contra a MÉDIA de n treinos: a diferença varia σ·√(1 + 1/n)
            tol = 2 * s * np.sqrt(1 + 1 / len(v))
            marca = "dentro de ± 2σ" if abs(ACC[k] - m) <= tol else "fora de ± 2σ"
            linha += f"{pct(m) + ' ± ' + br(100 * s) + ' %':>25s}   {marca}"
        elif len(v) == 1:
            linha += f"{pct(v[0]):>25s}   (1 semente: sem σ)"
        else:
            linha += f"{'a medir':>25s}"
        print(linha)
    if ref is None and REFERENCIA["completo"] is not None:
        print("Só para orientação, o modo completo do professor (outro protocolo, sem ± 2σ):")
        for k in chaves:
            print(f"  {NOMES[k]:18s}{pct(np.mean(REFERENCIA['completo'][k])):>9s}")

## Quanto tempo cada treino leva nesta máquina

Antes de treinar de verdade, a MLP e a CNN da aula fazem algumas dezenas de lotes com
fotos de ruído, só para medir esta máquina. Da duração de um lote sai a estimativa de
cada treino do notebook, sem contar a validação.

In [ ]:
def segundos_por_lote(modelo, n_lotes=40):
    """Duração de um passo de treino (um lote de LOTE fotos), com fotos de ruído."""
    rng = np.random.default_rng(0)
    x = rng.random((LOTE * n_lotes, LADO, LADO, 3), dtype=np.float32)
    y = rng.integers(0, 10, LOTE * n_lotes)
    modelo.fit(x[:LOTE * 4], y[:LOTE * 4], batch_size=LOTE, epochs=1, verbose=0)   # monta o grafo
    t0 = time.perf_counter()
    modelo.fit(x, y, batch_size=LOTE, epochs=1, verbose=0)
    return (time.perf_counter() - t0) / n_lotes


with cronometro("medir esta máquina"):
    S_LOTE = {"mlp": segundos_por_lote(compilar(construir_mlp())),
              "cnn": segundos_por_lote(compilar(construir_cnn()))}

N_PEQUENO = 10 * N_POR_CLASSE_PEQUENO
N_MODO = N_TREINO if MODO == "completo" else N_PEQUENO
treinos = [("MLP, seção 3", "mlp", N_MODO, EPOCAS, 2),
           ("CNN, seções 7 e 10", "cnn", N_MODO, EPOCAS, 2),
           ("CNN, seção 11", "cnn", N_PEQUENO, EPOCAS_SOBREAJUSTE, 2)]
total = 0.0
for rede in ("mlp", "cnn"):
    print(f"{rede.upper()}: {br(1000 * S_LOTE[rede], 0)} ms por lote de {LOTE} fotos")
for rotulo, rede, n, ep, vezes in treinos:
    t = S_LOTE[rede] * np.ceil(n / LOTE) * ep
    total += vezes * t
    print(f"{rotulo}: {vezes} treinos de ≈ {br(t / 60)} min ({br(n, 0)} fotos, {ep} épocas)")
print(f"total dos treinos, modo {MODO}: ≈ {br(total / 60)} min, sem a validação e sem os exercícios")

# 1 — Os dados

O CIFAR-10 (Krizhevsky, 2009) tem 60.000 fotos coloridas de 32 × 32 px em 10 classes:
50.000 de treino e 10.000 de teste.

### O arquivo do Teams

O Keras baixa o CIFAR-10 do servidor da Universidade de Toronto, que é lento: em
05/11/2026 foram 36 min para os 163 MiB do arquivo. O mesmo arquivo,
`cifar-10-python.tar.gz`, está no Teams da disciplina. Basta colocá-lo na pasta deste
notebook (no Colab, no painel de arquivos, em `/content`). A célula abaixo confere a
integridade dele pelo SHA-256 e o copia para o cache do Keras **com o nome que o Keras
procura**, `cifar-10-batches-py-target_archive`.

Para copiar à mão, o destino é `~/.keras/datasets/cifar-10-batches-py-target_archive` (no Windows,
`C:\Users\<usuário>\.keras\datasets\cifar-10-batches-py-target_archive`). Copiar com o nome original
não adianta: o Keras 3.15 não reconhece `cifar-10-python.tar.gz` e baixa tudo de novo.

In [ ]:
ARQUIVO_TEAMS = Path("cifar-10-python.tar.gz")          # ao lado deste notebook
SHA256_CIFAR = "6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce"
NOME_NO_CACHE = "cifar-10-batches-py-target_archive"       # o nome que o Keras 3.15 procura
CACHE_KERAS = Path(os.environ.get("KERAS_HOME", Path.home() / ".keras")) / "datasets"

if "cifar-10-batches-py-target" not in inspect.getsource(keras.datasets.cifar10.load_data):
    print("AVISO: esta versão do Keras guarda o CIFAR-10 com outro nome; a cópia abaixo "
          "pode não ser usada, e o Keras baixará de Toronto.")


def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


destino_cifar = CACHE_KERAS / NOME_NO_CACHE
if destino_cifar.exists():
    print("o CIFAR-10 já está no cache do Keras")
elif ARQUIVO_TEAMS.exists():
    assert sha256(ARQUIVO_TEAMS) == SHA256_CIFAR, "arquivo corrompido: baixe-o de novo do Teams"
    CACHE_KERAS.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(ARQUIVO_TEAMS, destino_cifar)
    print("copiado para o cache do Keras:", destino_cifar)
else:
    print("cifar-10-python.tar.gz não está nesta pasta: o Keras vai baixar de Toronto")

## Carregar uma única vez

O CIFAR-10 é carregado **uma vez** e reaproveitado por todo o notebook. A divisão é a
do protocolo: as 45.000 primeiras fotos de treino treinam; as 5.000 últimas
formam a **validação**, que acompanha o treino e escolhe entre modelos; as 10.000 de
**teste** só medem o resultado final de cada rede. No modo rápido, o treino usa
1.000 fotos de cada classe, sorteadas dentro das 45.000.

In [ ]:
with cronometro("carregar o CIFAR-10"):
    (x_tr, y_tr), (x_val, y_val), (x_te, y_te) = carregar_cifar10()
x_val = x_val.copy()     # sem a cópia, a validação prende na memória o bloco das 50.000 fotos
IDX_PEQUENO = subconjunto_pequeno(y_tr)          # as fotos do modo rápido e da seção 11
if MODO == "rapido":
    x_tr, y_tr = x_tr[IDX_PEQUENO], y_tr[IDX_PEQUENO]

for nome, x in (("treino", x_tr), ("validação", x_val), ("teste", x_te)):
    print(f"{nome:10s} {br(len(x), 0):>7s} fotos · {br(x.nbytes / 1e6, 0):>4s} MB em float32")
print("fotos de treino por classe:", np.bincount(y_tr, minlength=10))

Uma foto de cada classe: a primeira de cada classe no teste, a mesma regra da análise do
professor. Com 32 × 32 px, as fotos aparecem pixeladas; nenhuma suavização é aplicada.

In [ ]:
EXEMPLOS = [int(np.flatnonzero(y_te == c)[0]) for c in range(10)]


def grade(fotos, titulos, linhas=1, lado=1.3):
    """Fotos lado a lado, sem eixos e sem suavização."""
    colunas = int(np.ceil(len(fotos) / linhas))
    fig, eixos = plt.subplots(linhas, colunas, figsize=(lado * colunas, lado * linhas + 0.5),
                              squeeze=False)
    for ax in eixos.ravel():
        ax.set_axis_off()
    for ax, foto, titulo in zip(eixos.ravel(), fotos, titulos):
        ax.imshow(foto, interpolation="nearest")
        ax.set_title(titulo, fontsize=8)
    plt.tight_layout()
    plt.show()


grade(x_te[EXEMPLOS], [f"{CLASSES[c]}\nteste[{i}]" for c, i in enumerate(EXEMPLOS)])

# 2 — O enigma montado

Uma permutação das 1.024 posições, sorteada uma vez (semente 2026) e
aplicada a todas as fotos: o pixel que estava na posição `perm[q]` vai para a posição
`q`, com os três canais juntos. Nenhum número da foto é apagado ou alterado; só a ordem
muda. Por isso o embaralhamento se desfaz: `np.argsort(perm)` é a permutação inversa.

In [ ]:
perm = permutacao()
inversa = np.argsort(perm)
print("perm[:8] =", perm[:8])

x_te_emb = embaralhar(x_te, perm)
assert np.array_equal(embaralhar(x_te_emb, inversa), x_te)       # igualdade exata, bit a bit
# em cada foto, os mesmos 3.072 números, só em outra ordem
assert all(np.array_equal(np.sort(a.ravel()), np.sort(b.ravel()))
           for a, b in zip(x_te[:100], x_te_emb[:100]))
print(f"desembaralhar com argsort(perm) devolve as {br(len(x_te), 0)} fotos de teste, bit a bit")

In [ ]:
with cronometro("embaralhar treino e validação"):
    x_tr_emb = embaralhar(x_tr, perm)
    x_val_emb = embaralhar(x_val, perm)
memoria = (x_tr_emb.nbytes + x_val_emb.nbytes + x_te_emb.nbytes) / 1e6
print(f"memória das cópias embaralhadas: {br(memoria, 0)} MB")

As mesmas dez fotos, antes e depois. Para os olhos, o objeto some. A informação, porém,
está toda lá: a primeira célula desta seção desfez o embaralhamento sem perder nada.

In [ ]:
grade(np.concatenate([x_te[EXEMPLOS], x_te_emb[EXEMPLOS]]),
      [CLASSES[c] for c in range(10)] + ["embaralhada"] * 10, linhas=2)

# 3 — A MLP nas fotos e nas embaralhadas

A rede densa (MLP) achata a foto numa fila de 3.072 números (32 × 32 px × 3 canais) e
liga cada um deles a cada neurônio da primeira camada. O treino é o do protocolo: Adam
com taxa de aprendizado 10⁻³, lotes de 64 fotos sorteados de novo a cada
**época** (uma passada por todas as fotos de treino) e a validação medida no fim de
cada época. O teste é medido uma vez, depois do treino.

Nos treinos do modo completo, o professor ligou o determinismo das operações
(`enable_op_determinism`), que deixa o treino mais lento; este notebook não liga, como
os treinos de referência do modo rápido. Por isso, duas execuções da mesma célula dão
resultados um pouco diferentes.

In [ ]:
def treinar(modelo, x, y, x_val, y_val, epocas=EPOCAS, rotulo="treino", callbacks=()):
    """O treino do protocolo: lotes de LOTE fotos, ordem sorteada a cada época, validação
    no fim de cada época. Imprime o tempo e devolve o histórico do fit."""
    with cronometro(rotulo):
        historico = modelo.fit(x, y, epochs=epocas, batch_size=LOTE, shuffle=True,
                               validation_data=(x_val, y_val), callbacks=list(callbacks),
                               verbose=2)
    return historico


def acuracia_teste(modelo, x):
    """Acurácia nas fotos de teste x (as fotos ou as embaralhadas)."""
    return float(modelo.evaluate(x, y_te, batch_size=500, verbose=0)[1])

## 3.1 — Nas fotos

In [ ]:
mlp_fotos = compilar(construir_mlp())
mlp_fotos.summary()
assert mlp_fotos.count_params() == PARAMS_MLP

h_mlp_fotos = treinar(mlp_fotos, x_tr, y_tr, x_val, y_val, rotulo="treino da MLP nas fotos")
ACC["mlp_fotos"] = acuracia_teste(mlp_fotos, x_te)
print("acurácia no teste:", pct(ACC["mlp_fotos"]))

## 3.2 — Nas embaralhadas

Uma MLP nova, treinada do zero nas fotos embaralhadas e testada nas embaralhadas.

In [ ]:
mlp_emb = compilar(construir_mlp())
h_mlp_emb = treinar(mlp_emb, x_tr_emb, y_tr, x_val_emb, y_val,
                    rotulo="treino da MLP nas embaralhadas")
ACC["mlp_emb"] = acuracia_teste(mlp_emb, x_te_emb)
print("acurácia no teste:", pct(ACC["mlp_emb"]))

In [ ]:
comparar(["mlp_fotos", "mlp_emb"])

A diferença entre as duas MLPs é maior que o ruído de semente do professor (o σ da
tabela)? A seção seguinte responde sem treinar nada.

# 4 — A identidade: por que a MLP não percebe o embaralhamento

Cada neurônio k da primeira camada densa calcula uma soma ponderada dos 3.072 números
da fila. Se os números da entrada e as linhas da matriz de pesos forem permutados **da
mesma forma**, cada produto continua na soma, só em outra posição:

$$z_k = \sum_{q} x^{\text{emb}}_{q}\, w^{\text{emb}}_{qk}
      = \sum_{q} x_{\pi(q)}\, w_{\pi(q)k} = \sum_{p} x_p\, w_{pk}.$$

Logo, a MLP treinada nas fotos, com as linhas do kernel da primeira camada permutadas,
dá nas fotos embaralhadas **as mesmas previsões** que dava nas fotos, sem treino
nenhum. É a **rota 1** da aula. Cada pixel ocupa 3 linhas do kernel (R, G e B), que
viajam juntas. A **rota 2** é a MLP da seção 3.2, treinada do zero nas embaralhadas.
As duas rotas só podem discordar pelo ruído de semente.

In [ ]:
K, b = mlp_fotos.get_layer("densa1").get_weights()              # K: 3.072 linhas × 512 neurônios
K_e = K.reshape(1024, 3, -1)[perm].reshape(3072, -1)            # as 3 linhas de cada pixel juntas
mlp_rota1 = construir_mlp()
mlp_rota1.set_weights([K_e, b] + mlp_fotos.get_weights()[2:])

p_fotos = mlp_fotos.predict(x_te, batch_size=500, verbose=0)
p_rota1 = mlp_rota1.predict(x_te_emb, batch_size=500, verbose=0)
iguais = int(np.sum(p_fotos.argmax(1) == p_rota1.argmax(1)))
ACC["mlp_rota1"] = float(np.mean(p_rota1.argmax(1) == y_te))
print(f"previsões iguais: {br(iguais, 0)} de {br(len(x_te), 0)} fotos")
print(f"maior diferença numa probabilidade: {np.abs(p_fotos - p_rota1).max():.1e}")
print("acurácia da rota 1 nas embaralhadas:", pct(ACC["mlp_rota1"], 2),
      "· da MLP nas fotos:", pct(ACC["mlp_fotos"], 2))

Em float32, a permutação muda a ordem da soma, e o arredondamento pode desempatar de
outro jeito uma foto em que as duas maiores probabilidades estão empatadas. A mesma
conta em float64, em numpy e em blocos de 1.000 fotos, tira a dúvida: as previsões têm
de ser todas iguais.

In [ ]:
W = [w.astype(np.float64) for w in mlp_fotos.get_weights()]
W_e = [K_e.astype(np.float64)] + W[1:]


def classe_float64(pesos, x):
    h = x.reshape(len(x), -1).astype(np.float64)
    for i in range(0, 6, 2):
        h = h @ pesos[i] + pesos[i + 1]
        if i < 4:
            h = np.maximum(h, 0)          # ReLU nas duas camadas escondidas
    return h.argmax(1)


iguais64 = sum(int(np.sum(classe_float64(W, x_te[i:i + 1000]) ==
                          classe_float64(W_e, x_te_emb[i:i + 1000])))
               for i in range(0, len(x_te), 1000))
assert iguais64 == len(x_te)
print(f"float64: {br(iguais64, 0)} previsões iguais em {br(len(x_te), 0)}")

## As duas rotas

Aqui há uma semente de cada lado: a diferença entre as rotas é a diferença entre duas
sementes. Nos slides, com várias sementes de cada lado, as médias das duas rotas são
comparadas com o erro-padrão delas.

In [ ]:
print("rota 1 (MLP das fotos, kernel permutado):", pct(ACC["mlp_rota1"]))
print("rota 2 (MLP treinada nas embaralhadas):  ", pct(ACC["mlp_emb"]))
dif = abs(ACC["mlp_rota1"] - ACC["mlp_emb"])
ref = REFERENCIA[MODO]
if ref is not None and len(ref["mlp_emb"]) >= 2:
    s1, s2 = np.std(ref["mlp_fotos"], ddof=1), np.std(ref["mlp_emb"], ddof=1)
    print(f"diferença: {br(100 * dif)} pontos percentuais; σ do professor no modo {MODO}: "
          f"{br(100 * s1)} (MLP-fotos) e {br(100 * s2)} pontos percentuais (MLP-embaralhadas)")
else:
    print(f"diferença: {br(100 * dif)} pontos percentuais; ruído de semente deste modo: a medir")

r = REF_ANALISE["rotas"]
n = len(REF_ANALISE["rota1_iguais_float32"])
print(f"\nprofessor, modo completo, {n} sementes de cada lado:")
print(f"  rota 1: {pct(r['A1'], 2)} ± {br(100 * r['sigma1'], 2)} % · "
      f"rota 2: {pct(r['A2'], 2)} ± {br(100 * r['sigma2'], 2)} %")
print(f"  |diferença| = {br(100 * r['dif'], 2)} pontos percentuais; limite de 2 erros-padrão: "
      f"{br(100 * r['limite'], 2)} → {'concordam' if r['passa'] else 'NÃO concordam'}")

# 5 — Vizinhos se parecem: a aposta da CNN

A CNN olha só 3 × 3 vizinhos de cada vez e usa os mesmos pesos em toda a foto. Isso só
compensa se pixels próximos tiverem algo em comum. A medida direta é a correlação r(d)
entre a luminância de dois pixels a d px um do outro na mesma linha, juntando as 10.000
fotos de teste, nas fotos e nas embaralhadas. A conta é a mesma da análise do
professor, e o resultado tem de ser o mesmo: não há treino nem sorteio no caminho.

In [ ]:
LUM = np.array([0.299, 0.587, 0.114])               # pesos da luminância (Rec. 601)
lum = np.tensordot(x_te, LUM.astype(np.float32), axes=([3], [0]))           # 10.000 × 32 × 32
lum_emb = np.tensordot(x_te_emb, LUM.astype(np.float32), axes=([3], [0]))
D = np.arange(1, 17)                                                          # px


def r_vizinhos(L, d):
    """Correlação entre pixels a d px na horizontal, juntando todas as fotos."""
    return float(np.corrcoef(L[:, :, :-d].ravel(), L[:, :, d:].ravel())[0, 1])


with cronometro("medir r(d)"):
    r_fotos = [r_vizinhos(lum, d) for d in D]
    r_emb = [r_vizinhos(lum_emb, d) for d in D]

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(D, r_fotos, "o-", color="k", label="fotos")
ax.plot(D, r_emb, "s--", color="0.5", label="embaralhadas")
ax.set_xlabel("distância d (px)"); ax.set_ylabel("correlação r(d)")
ax.set_ylim(-0.1, 1); ax.legend()
plt.tight_layout(); plt.show()
print(f"d = 1 px: fotos {br(r_fotos[0], 3)} · embaralhadas {br(r_emb[0], 3)}")
assert np.allclose(r_fotos, REF_ANALISE["correlacao"]["fotos_h"], atol=1e-4)
assert np.allclose(r_emb, REF_ANALISE["correlacao"]["emb_h"], atol=1e-4)
print("iguais aos da análise do professor a 4 casas")

Nas embaralhadas, r(d) não vai a zero. O motivo é o brilho de cada foto: numa foto
clara, todos os pixels são claros, e dois pixels quaisquer da mesma foto se
correlacionam, vizinhos ou não. Tirando de cada foto o próprio brilho médio, sobra só
o que depende da vizinhança.

In [ ]:
lum_c = lum - lum.mean(axis=(1, 2), keepdims=True)            # cada foto menos o próprio brilho
lum_emb_c = lum_emb - lum_emb.mean(axis=(1, 2), keepdims=True)
r_fotos_c = [r_vizinhos(lum_c, d) for d in D]
r_emb_c = [r_vizinhos(lum_emb_c, d) for d in D]

fig, eixos = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (rf, re, titulo) in zip(eixos, ((r_fotos, r_emb, "cada foto com o próprio brilho"),
                                        (r_fotos_c, r_emb_c, "brilho médio de cada foto subtraído"))):
    ax.plot(D, rf, "o-", color="k", label="fotos")
    ax.plot(D, re, "s--", color="0.5", label="embaralhadas")
    ax.axhline(0, color="0.8", lw=0.8)
    ax.set_title(titulo, fontsize=9); ax.set_xlabel("distância d (px)")
eixos[0].set_ylabel("correlação r(d)"); eixos[0].legend()
plt.tight_layout(); plt.show()
print(f"centrada, d = 1 px: fotos {br(r_fotos_c[0], 3)} · embaralhadas {br(r_emb_c[0], 3)}")
assert np.allclose(r_fotos_c, REF_ANALISE["correlacao"]["fotos_h_centrada"], atol=1e-4)
assert np.allclose(r_emb_c, REF_ANALISE["correlacao"]["emb_h_centrada"], atol=1e-4)

Outra forma de ver o estrago: onde estavam, na foto original, os pixels que viraram
vizinhos depois do embaralhamento? Para cada par de vizinhos da foto embaralhada (na
horizontal, na vertical e nas diagonais), a distância entre as posições de origem.

In [ ]:
pos = np.stack(np.unravel_index(np.arange(LADO * LADO), (LADO, LADO)), 1).astype(float)
pares = []
for i in range(LADO):
    for j in range(LADO):
        for di, dj in ((0, 1), (1, 0), (1, 1), (1, -1)):
            if 0 <= i + di < LADO and 0 <= j + dj < LADO:
                pares.append((LADO * i + j, LADO * (i + di) + j + dj))
pares = np.array(pares)
d_fotos = np.linalg.norm(pos[pares[:, 0]] - pos[pares[:, 1]], axis=1)
d_emb = np.linalg.norm(pos[perm[pares[:, 0]]] - pos[perm[pares[:, 1]]], axis=1)   # a posição q guarda o pixel perm[q]
todos = np.linalg.norm(pos[:, None] - pos[None], axis=2)
acaso = todos[np.triu_indices(LADO * LADO, 1)].mean()
print(f"{br(len(pares), 0)} pares de vizinhos")
print(f"distância média na foto: vizinhos de verdade {br(d_fotos.mean(), 2)} px · "
      f"vizinhos depois do embaralhamento {br(d_emb.mean(), 2)} px · "
      f"dois pixels sorteados ao acaso {br(acaso, 2)} px")
assert abs(d_emb.mean() - REF_ANALISE["distancias"]["vizinhos_embaralhados_px"]) < 1e-6

# 6 — Uma camada convolucional à mão

A Aula 07 deslizou um kernel 3 × 3 sobre uma imagem em tons de cinza. Numa foto
colorida, o kernel ganha a profundidade dos canais, 3 × 3 × 3, e cada posição do
**mapa** de saída soma 27 produtos e o viés:

$$y[i, j] = b + \sum_{c=1}^{3} \sum_{u=-1}^{1} \sum_{v=-1}^{1} w[u, v, c]\;
x_c[i + u,\, j + v].$$

O kernel não é espelhado: a "convolução" das redes é a correlação da Aula 07. O laço
abaixo faz a conta para um kernel sorteado, no navio da aula, e confere contra uma
camada `Conv2D` do Keras carregada com os mesmos pesos.

In [ ]:
HEROI_FOTO = x_te[HEROI]
assert CLASSES[y_te[HEROI]] == "navio"
rng = np.random.default_rng(0)
w = rng.normal(0, 0.3, (3, 3, 3)).astype(np.float32)     # linhas × colunas × canais
b0 = np.float32(0.1)

# à mão, com padding "same": uma moldura de 1 px de zeros em volta da foto
moldura = np.pad(HEROI_FOTO, ((1, 1), (1, 1), (0, 0)))
mapa = np.zeros((LADO, LADO), np.float32)
for i in range(LADO):
    for j in range(LADO):
        mapa[i, j] = np.sum(w * moldura[i:i + 3, j:j + 3, :]) + b0

conv = layers.Conv2D(1, 3, padding="same")
conv.build((None, LADO, LADO, 3))
conv.set_weights([w[..., None], np.array([b0])])
mapa_keras = keras.ops.convert_to_numpy(conv(HEROI_FOTO[None]))[0, :, :, 0]
dif = float(np.abs(mapa - mapa_keras).max())
assert dif < 1e-5
print(f"maior |Δ| entre o laço e a Conv2D: {dif:.1e}")

lim = np.abs(mapa).max()
fig, eixos = plt.subplots(1, 3, figsize=(8, 2.8))
eixos[0].imshow(HEROI_FOTO, interpolation="nearest"); eixos[0].set_title(f"teste[{HEROI}], 32 × 32 px", fontsize=9)
eixos[1].imshow(mapa, cmap="PuOr", vmin=-lim, vmax=lim, interpolation="nearest"); eixos[1].set_title("o laço", fontsize=9)
eixos[2].imshow(mapa_keras, cmap="PuOr", vmin=-lim, vmax=lim, interpolation="nearest"); eixos[2].set_title("a Conv2D", fontsize=9)
for ax in eixos:
    ax.set_axis_off()
plt.tight_layout(); plt.show()

## Padding e stride

Com `padding="valid"`, o kernel só para onde cabe inteiro, e o mapa encolhe; com
`padding="same"`, uma moldura de zeros mantém o tamanho. Com `strides=2` (o **passo**,
ou *stride*), o kernel pula de 2 em 2 px. Para uma entrada de n px, kernel de k px,
moldura de p px e passo s:

$$n_{\text{saída}} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1.$$

Com passo 2, o `"same"` do Keras põe a moldura que faltar, às vezes só de um lado; a
fórmula com p = 1 dá o mesmo tamanho.

In [ ]:
def n_saida(n, k, p, s):
    return (n + 2 * p - k) // s + 1


print(" n (px)  padding  passo   Keras (px)   fórmula (px)")
for n in (7, LADO):
    for padding, p in (("valid", 0), ("same", 1)):
        for s in (1, 2):
            forma = layers.Conv2D(4, 3, padding=padding, strides=s)(np.zeros((1, n, n, 3), np.float32)).shape
            assert forma[1] == forma[2] == n_saida(n, 3, p, s)
            print(f"{n:6d}   {padding:7s} {s:5d} {forma[1]:12d} {n_saida(n, 3, p, s):14d}")

## Duas convoluções seguidas, sem nada no meio, valem por uma

Duas correlações 3 × 3 seguidas são uma correlação só, com um kernel 5 × 5: a
convolução completa dos dois kernels. A igualdade vale no interior do mapa; na borda, a
moldura de zeros de cada camada a quebra. Com uma ReLU entre as duas, a igualdade se
perde: é a não linearidade que faz a profundidade valer alguma coisa. Na CNN da aula,
além da ReLU, há um *pooling* entre as convoluções.

In [ ]:
from scipy.signal import convolve2d, correlate2d

media = np.full((3, 3), 1 / 9)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float)
cinza = 255 * np.tensordot(HEROI_FOTO.astype(np.float64), LUM, axes=([2], [0]))   # níveis de cinza
composto = convolve2d(media, sobel_y, "full")                   # 5 × 5
print("kernel composto × 9:\n", np.round(9 * composto).astype(int))

duas = correlate2d(correlate2d(cinza, media, "valid"), sobel_y, "valid")
uma = correlate2d(cinza, composto, "valid")
print(f"duas camadas × uma, sem moldura: maior |Δ| = {np.abs(duas - uma).max():.1e} níveis de cinza")

dif_same = np.abs(correlate2d(correlate2d(cinza, media, "same"), sobel_y, "same")
                  - correlate2d(cinza, composto, "same"))
print(f"com moldura de zeros: interior {dif_same[2:-2, 2:-2].max():.1e}, "
      f"borda até {br(dif_same.max())} níveis de cinza")

com_relu = correlate2d(np.maximum(correlate2d(cinza, sobel_y, "valid"), 0), media, "valid")
print(f"com uma ReLU no meio: maior |Δ| = {br(np.abs(com_relu - uma).max())} níveis de cinza")

# 7 — A CNN da aula

Três blocos iguais, cada um com uma convolução 3 × 3 (`padding="same"`), uma ReLU e um
*max pooling* 2 × 2, que guarda o maior valor de cada bloco de 2 × 2 px e reduz o mapa
à metade; depois, a fila, uma camada densa de 128 neurônios e a saída com 10
probabilidades (softmax). A ReLU é uma camada própria, e não um `activation=` da
`Conv2D`, para que o mapa possa ser lido antes e depois dela (seção 8).

Cada convolução tem (3 · 3 · C_entrada + 1) · K **parâmetros** (os números que o treino
ajusta): K kernels de 3 × 3 × C_entrada pesos, mais um viés por kernel. A célula
confere a conta, camada a camada, contra o `summary()`.

In [ ]:
cnn_fotos = compilar(construir_cnn())
cnn_fotos.summary()

c_entrada = 3
for camada in cnn_fotos.layers:
    if isinstance(camada, layers.Conv2D):
        kh, kw = camada.kernel_size
        conta = (kh * kw * c_entrada + 1) * camada.filters
        assert camada.count_params() == conta == PARAMS_CNN_CAMADAS[camada.name]
        print(f"{camada.name}: ({kh}·{kw}·{c_entrada} + 1)·{camada.filters} = {br(conta, 0)} parâmetros")
        c_entrada = camada.filters
n_fila = int(np.prod(cnn_fotos.get_layer("fila").output.shape[1:]))
for nome, n_entrada in (("densa", n_fila), ("saida", cnn_fotos.get_layer("densa").units)):
    unidades = cnn_fotos.get_layer(nome).units
    conta = (n_entrada + 1) * unidades
    assert cnn_fotos.get_layer(nome).count_params() == conta == PARAMS_CNN_CAMADAS[nome]
    print(f"{nome}: ({br(n_entrada, 0)} + 1)·{unidades} = {br(conta, 0)} parâmetros")
assert cnn_fotos.count_params() == PARAMS_CNN
print(f"total: {br(cnn_fotos.count_params(), 0)} parâmetros "
      f"(a MLP tem {br(PARAMS_MLP, 0)})")

## O treino

O mesmo treino da MLP. O modelo treinado fica em `saida/cnn_fotos.keras`.

In [ ]:
h_cnn_fotos = treinar(cnn_fotos, x_tr, y_tr, x_val, y_val, rotulo="treino da CNN nas fotos")
ACC["cnn_fotos"] = acuracia_teste(cnn_fotos, x_te)
cnn_fotos.save(SAIDA / "cnn_fotos.keras")
print("acurácia no teste:", pct(ACC["cnn_fotos"]))

A acurácia de treino que o `fit` relata é a média corrida da época: mistura os pesos do
começo e do fim dela. A de validação é medida com os pesos do fim da época.

In [ ]:
ep = np.arange(1, len(h_cnn_fotos.history["accuracy"]) + 1)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(ep, 100 * np.array(h_cnn_fotos.history["accuracy"]), "o-", color="0.55",
        label="treino (média corrida da época)")
ax.plot(ep, 100 * np.array(h_cnn_fotos.history["val_accuracy"]), "s-", color="k", label="validação")
ax.set_xlabel("época"); ax.set_ylabel("acurácia (%)"); ax.legend()
plt.tight_layout(); plt.show()

# 8 — Dentro da CNN

## Os 32 kernels da primeira camada

Cada kernel da conv1 tem 3 × 3 × 3 pesos. Em cores, cada kernel é esticado para ocupar
a escala inteira; em cinza, a soma dos três canais ponderada pela luminância mostra o
sinal: escuro é peso negativo, claro é positivo. Algum lembra os kernels de borda da
Aula 07, como o Sobel?

In [ ]:
K1, _ = cnn_fotos.get_layer("conv1").get_weights()       # 3 × 3 × 3 × 32
fig, eixos = plt.subplots(4, 16, figsize=(12, 3.6))
for k in range(K1.shape[-1]):
    rgb = K1[..., k]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    k_cinza = np.tensordot(K1[..., k], LUM, axes=([2], [0]))
    m = np.abs(k_cinza).max()
    linha, coluna = 2 * (k // 16), k % 16
    eixos[linha, coluna].imshow(rgb, interpolation="nearest")
    eixos[linha, coluna].set_title(str(k), fontsize=7)
    eixos[linha + 1, coluna].imshow(k_cinza, cmap="gray", vmin=-m, vmax=m, interpolation="nearest")
for ax in eixos.ravel():
    ax.set_axis_off()
fig.suptitle("conv1: os 32 kernels de 3 × 3 px, em cores (linhas 1 e 3) e em cinza (linhas 2 e 4)", fontsize=9)
plt.tight_layout(); plt.show()

## O navio, camada a camada

Um `keras.Model` com a mesma entrada da CNN e uma saída por camada devolve todos os
mapas intermediários de uma vez. Para cada bloco, os 4 mapas de maior variância depois
da ReLU (a regra da análise do professor), vistos antes da ReLU, depois dela e depois do
*pooling*. Mapas com sinal aparecem em escala divergente, centrada no zero; mapas sem
valores negativos, em cinza.

In [ ]:
NOMES_CAMADAS = ["conv1", "relu1", "pool1", "conv2", "relu2", "pool2", "conv3", "relu3", "pool3"]
por_dentro = keras.Model(cnn_fotos.inputs, [cnn_fotos.get_layer(n).output for n in NOMES_CAMADAS])
mapas = dict(zip(NOMES_CAMADAS, (m[0] for m in por_dentro.predict(x_te[HEROI:HEROI + 1], verbose=0))))
ordem = {n[-1]: np.argsort(-mapas[n].reshape(-1, mapas[n].shape[-1]).var(0))[:4]
         for n in ("relu1", "relu2", "relu3")}

fig, eixos = plt.subplots(len(NOMES_CAMADAS), 5, figsize=(7.5, 13))
for linha, nome in enumerate(NOMES_CAMADAS):
    m = mapas[nome]
    eixos[linha, 0].text(0, 0.5, f"{nome}\n{m.shape[0]} × {m.shape[1]} px\n{m.shape[2]} mapas",
                         va="center", fontsize=9)
    for coluna, k in enumerate(ordem[nome[-1]], start=1):
        ax = eixos[linha, coluna]
        if nome.startswith("conv"):
            lim = np.abs(m[..., k]).max()
            ax.imshow(m[..., k], cmap="PuOr", vmin=-lim, vmax=lim, interpolation="nearest")
        else:
            ax.imshow(m[..., k], cmap="gray", vmin=0, interpolation="nearest")
        ax.set_title(f"mapa {k}", fontsize=7)
for ax in eixos.ravel():
    ax.set_axis_off()
plt.tight_layout(); plt.show()

## Campo receptivo

Cada valor de um mapa depende só de uma janela da foto: o **campo receptivo** dele.
Para medi-lo sem conta à mão, a rede é copiada com as mesmas formas, mas com kernels de
uns, sem viés, sem ReLU e com média no lugar do máximo; com tudo positivo, nada se
cancela. Um delta (um único pixel aceso) em cada uma das 1.024 posições passa pela
cópia, e o campo receptivo do valor central de cada mapa é o conjunto das posições
cujo delta chega até ele.

In [ ]:
def copia_de_uns(modelo):
    """As camadas de convolução e de pooling de `modelo`, com as mesmas formas, kernels de
    uns, sem viés, sem ReLU e com média no lugar do máximo."""
    camadas = [keras.Input((LADO, LADO, 3))]
    for c in modelo.layers:
        if isinstance(c, layers.Flatten):
            break
        cfg = c.get_config()
        if isinstance(c, layers.Conv2D):
            camadas.append(layers.Conv2D(1, cfg["kernel_size"], strides=cfg["strides"],
                                         padding=cfg["padding"], use_bias=False,
                                         kernel_initializer="ones", name=c.name))
        elif isinstance(c, (layers.MaxPooling2D, layers.AveragePooling2D)):
            camadas.append(layers.AveragePooling2D(cfg["pool_size"], strides=cfg["strides"],
                                                   padding=cfg["padding"], name=c.name))
    return keras.Sequential(camadas)


def campo_receptivo(modelo):
    """Lado (px) e canto (linha, coluna) do campo receptivo do valor central de cada mapa."""
    uns = copia_de_uns(modelo)
    nomes = [c.name for c in uns.layers]
    saidas = keras.Model(uns.inputs, [uns.get_layer(n).output for n in nomes])
    n = LADO * LADO
    deltas = np.zeros((n, LADO, LADO, 3), np.float32)
    deltas[np.arange(n), np.arange(n) // LADO, np.arange(n) % LADO, :] = 1
    lados, cantos = {}, {}
    for nome, o in zip(nomes, saidas.predict(deltas, batch_size=256, verbose=0)):
        c = o.shape[1] // 2
        janela = (o[:, c, c, 0] > 0).reshape(LADO, LADO)
        linhas, colunas = np.flatnonzero(janela.any(1)), np.flatnonzero(janela.any(0))
        lado = int(linhas[-1] - linhas[0] + 1)
        assert colunas[-1] - colunas[0] + 1 == lado and janela.sum() == lado ** 2
        lados[nome], cantos[nome] = lado, (int(linhas[0]), int(colunas[0]))
    return lados, cantos


CR, CR_CANTO = campo_receptivo(cnn_fotos)
print(" · ".join(f"{n}: {v} px" for n, v in CR.items()))
assert CR == REF_ANALISE["campo_receptivo_px"]

fig, ax = plt.subplots(figsize=(3.8, 3.8))
ax.imshow(x_te[HEROI], interpolation="nearest")
for nome, estilo in (("conv1", "-"), ("conv2", "--"), ("conv3", ":")):
    (i0, j0), lado = CR_CANTO[nome], CR[nome]
    ax.add_patch(plt.Rectangle((j0 - 0.5, i0 - 0.5), lado, lado, fill=False, color="w",
                               lw=2, ls=estilo, label=f"{nome}: {lado} × {lado} px"))
ax.legend(fontsize=7, loc="lower left"); ax.set_axis_off()
ax.set_title("campo receptivo do valor central", fontsize=9)
plt.tight_layout(); plt.show()

## O que os mapas da conv3 procuram

Para um mapa da conv3 depois da ReLU, as 9 fotos de teste em que ele atinge o maior
valor, e o recorte de 18 × 18 px em volta desse valor: o campo
receptivo dele. O mapa escolhido é o de maior variância do máximo por foto, a regra da
análise do professor. As partes do recorte fora da foto aparecem em branco.

In [ ]:
relu3 = keras.Model(cnn_fotos.inputs, cnn_fotos.get_layer("relu3").output)
lado_mapa, n_mapas = relu3.output.shape[1], relu3.output.shape[-1]      # 8 px, 128 mapas
maximo = np.zeros((len(x_te), n_mapas), np.float32)
onde = np.zeros((len(x_te), n_mapas), int)
for i in range(0, len(x_te), 500):
    o = relu3.predict(x_te[i:i + 500], verbose=0)
    o = o.reshape(len(o), -1, n_mapas)
    maximo[i:i + 500], onde[i:i + 500] = o.max(1), o.argmax(1)

k = int(np.argmax(maximo.var(0)))
passo = LADO // lado_mapa                                   # px da foto por posição do mapa
rf = CR["conv3"]
assert passo * (lado_mapa // 2) - (rf - passo) // 2 == CR_CANTO["conv3"][0]
recortes, titulos = [], []
for t in np.argsort(-maximo[:, k])[:9]:
    r, c = divmod(int(onde[t, k]), lado_mapa)
    y0, x0 = passo * r - (rf - passo) // 2, passo * c - (rf - passo) // 2
    rec = np.ones((rf, rf, 3), np.float32)
    ya, yb, xa, xb = max(y0, 0), min(y0 + rf, LADO), max(x0, 0), min(x0 + rf, LADO)
    rec[ya - y0:yb - y0, xa - x0:xb - x0] = x_te[t, ya:yb, xa:xb]
    recortes.append(rec)
    titulos.append(f"{CLASSES[y_te[t]]}\nteste[{t}]")
print(f"conv3, mapa {k}: recortes de {rf} × {rf} px")
grade(recortes, titulos)

## As probabilidades: o navio e um gato

A saída da rede é uma **softmax**: 10 probabilidades que somam 1. A **perda** de uma foto
é −ln da probabilidade da classe certa: perto de 0 quando a rede acerta com folga,
grande quando ela erra com confiança. O gato é a primeira foto de gato do teste.

In [ ]:
GATO = int(np.flatnonzero(y_te == CLASSES.index("gato"))[0])
probs = cnn_fotos.predict(x_te[[HEROI, GATO]], verbose=0)
fig, eixos = plt.subplots(1, 4, figsize=(11, 2.8), gridspec_kw={"width_ratios": [1, 2.2, 1, 2.2]})
for j, (i, p) in enumerate(zip((HEROI, GATO), probs)):
    eixos[2 * j].imshow(x_te[i], interpolation="nearest")
    eixos[2 * j].set_axis_off()
    eixos[2 * j].set_title(f"teste[{i}]", fontsize=8)
    ax = eixos[2 * j + 1]
    ax.barh(CLASSES, 100 * p, color="0.6")
    ax.invert_yaxis(); ax.set_xlim(0, 100)
    ax.set_xlabel("probabilidade (%)", fontsize=8); ax.tick_params(labelsize=7)
    perda = -np.log(max(float(p[y_te[i]]), 1e-12))
    ax.set_title(f"{CLASSES[y_te[i]]}: perda = −ln p = {br(perda, 3)}", fontsize=8)
plt.tight_layout(); plt.show()
print(f"perda de um chute uniforme (p = 1/10 em cada classe): {br(np.log(10), 3)}")

# 9 — Onde a rede erra

A matriz de confusão conta, para cada classe verdadeira (linha), quantas fotos de
teste foram para cada classe prevista (coluna). A diagonal são os acertos; o maior
número fora dela é o par que a rede mais confunde.

In [ ]:
from sklearn.metrics import confusion_matrix

pred = cnn_fotos.predict(x_te, batch_size=500, verbose=0).argmax(1)
conf = confusion_matrix(y_te, pred, labels=range(10))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(conf, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, conf[i, j], ha="center", va="center", fontsize=7,
                color="white" if conf[i, j] > conf.max() / 2 else "black")
ax.set_xticks(range(10), CLASSES, rotation=60, ha="right")
ax.set_yticks(range(10), CLASSES)
ax.set_xlabel("classe prevista"); ax.set_ylabel("classe verdadeira")
ax.set_title("CNN nas fotos: número de fotos de teste", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
acc_classe = np.diag(conf) / conf.sum(1)
prof = REF_ANALISE["confusao"]
print(f"{'classe':10s}{'aqui':>9s}{'professor':>12s}   (professor: semente 0, modo completo)")
for c in np.argsort(acc_classe):
    print(f"{CLASSES[c]:10s}{pct(acc_classe[c]):>9s}{pct(prof['acc_por_classe'][c]):>12s}")

fora = conf.copy()
np.fill_diagonal(fora, -1)
a, b = np.unravel_index(fora.argmax(), fora.shape)
print(f"\npar mais confundido: {CLASSES[a]} → {CLASSES[b]} ({conf[a, b]} fotos)")
pa, pb = prof["par"]
print(f"no professor: {CLASSES[pa]} → {CLASSES[pb]} ({prof['contagem']} fotos)")
exemplo = int(np.flatnonzero((y_te == a) & (pred == b))[0])
grade([x_te[exemplo]], [f"teste[{exemplo}]: {CLASSES[a]},\nprevista {CLASSES[b]}"], lado=2.4)

# 10 — A CNN nas embaralhadas

A CNN aposta que vizinhos se parecem (seção 5): olha 3 × 3 px de cada vez e reaproveita
os mesmos pesos em toda a foto. Nas embaralhadas, os vizinhos de um pixel vêm de
qualquer lugar da foto, e a aposta deixa de valer. A MLP não fazia aposta nenhuma
(seção 4). A mesma CNN, treinada do zero nas embaralhadas:

In [ ]:
cnn_emb = compilar(construir_cnn())
h_cnn_emb = treinar(cnn_emb, x_tr_emb, y_tr, x_val_emb, y_val, rotulo="treino da CNN nas embaralhadas")
ACC["cnn_emb"] = acuracia_teste(cnn_emb, x_te_emb)
print("acurácia no teste:", pct(ACC["cnn_emb"]))

## As quatro acurácias

A tabela põe as quatro acurácias deste notebook ao lado das do professor no mesmo modo.
No gráfico, as barras são as daqui, e o traço preto é a média do professor com a faixa
de ± 2σ. A tabela também vai para `saida/resultados.json`.

In [ ]:
ORDEM = ["mlp_fotos", "mlp_emb", "cnn_fotos", "cnn_emb"]
comparar(ORDEM)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
xs = np.arange(len(ORDEM))
ax.bar(xs, [100 * ACC[k] for k in ORDEM], color="0.8", edgecolor="k", label="este notebook")
ref = REFERENCIA[MODO]
if ref is not None and min(len(ref[k]) for k in ORDEM) >= 2:
    ax.errorbar(xs, [100 * np.mean(ref[k]) for k in ORDEM],
                yerr=[200 * np.std(ref[k], ddof=1) for k in ORDEM],
                fmt="o", color="k", capsize=6, label="professor: média ± 2σ")
ax.axhline(100 / len(CLASSES), ls=":", color="0.4", label="acaso: 1 classe em 10")
ax.set_xticks(xs, [NOMES[k] for k in ORDEM], fontsize=8)
ax.set_ylabel("acurácia no teste (%)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

resultado = {"modo": MODO, "acuracia_teste": ACC, "keras": keras.__version__,
             "tensorflow": tf.__version__}
(SAIDA / "resultados.json").write_text(json.dumps(resultado, indent=1, ensure_ascii=False))

**O enigma.** Qual das duas redes perdeu com o embaralhamento? A queda da CNN é maior
que o ruído de semente? E a CNN nas embaralhadas ficou acima, no nível ou abaixo da MLP
nas embaralhadas? A resposta está nas duas restrições da camada convolucional: cada
saída só vê 3 × 3 vizinhos, e os mesmos pesos valem em toda a foto. Elas são uma aposta
na vizinhança. Embaralhar desfaz a aposta, e o que resta é uma rede com restrições que
já não ajudam.

# 11 — Sobreajuste, aumento de dados e *callbacks*

Com poucas fotos e muitas épocas, a rede passa a decorar o treino: a acurácia de treino
continua subindo, e a de validação para ou cai. É o **sobreajuste**. Para vê-lo com
clareza, esta seção treina a CNN da aula com só 10.000 fotos (1.000
por classe), por 15 épocas no modo rápido ou 40 no completo,
que é o protocolo do vídeo.

A acurácia de treino que o `fit` relata é uma média corrida durante a época, e com o
aumento de dados ligado ela é medida em fotos alteradas. Para medir a memorização, um
*callback* (uma função que o `fit` chama no fim de cada época) avalia as fotos de treino
**sem** aumento: o `evaluate` desliga as camadas aleatórias.

In [ ]:
if MODO == "rapido":
    x_peq, y_peq = x_tr, y_tr                      # já são as fotos do subconjunto
else:
    x_peq, y_peq = x_tr[IDX_PEQUENO], y_tr[IDX_PEQUENO]
print(f"{br(len(x_peq), 0)} fotos de treino, {EPOCAS_SOBREAJUSTE} épocas")


class AcuraciaDeTreino(keras.callbacks.Callback):
    """No fim de cada época, a acurácia nas fotos de treino, sem aumento."""

    def __init__(self, x, y):
        super().__init__()
        self.x, self.y, self.acc = x, y, []

    def on_epoch_end(self, epoca, logs=None):
        self.acc.append(float(self.model.evaluate(self.x, self.y, batch_size=500, verbose=0)[1]))

## Sem aumento

O `ModelCheckpoint` grava em `saida/melhor_sem_aumento.keras` o modelo da época de menor
perda na **validação**. O teste não participa de nenhuma escolha.

In [ ]:
cnn_sem = compilar(construir_cnn())
med_sem = AcuraciaDeTreino(x_peq, y_peq)
guardar_sem = keras.callbacks.ModelCheckpoint(SAIDA / "melhor_sem_aumento.keras",
                                              monitor="val_loss", save_best_only=True)
h_sem = treinar(cnn_sem, x_peq, y_peq, x_val, y_val, epocas=EPOCAS_SOBREAJUSTE,
                rotulo="treino sem aumento", callbacks=[med_sem, guardar_sem])

## Com aumento de dados

O **aumento de dados** cria, a cada época, versões novas de cada foto de treino:
espelhada na horizontal e deslocada até 10 % do lado (3,2 px) em cada eixo, com a borda
refletida. No Keras 3, ele é feito por camadas no começo da rede (`RandomFlip` e
`RandomTranslation`, pedidas por `construir_cnn(aumento=True)`); elas só agem no `fit`
e deixam a foto intacta no `evaluate` e no `predict`. Não acrescentam parâmetros. Oito
sorteios das duas camadas, aplicados ao navio:

In [ ]:
cnn_aum = compilar(construir_cnn(aumento=True))
assert cnn_aum.count_params() == PARAMS_CNN
variantes = np.repeat(x_te[HEROI:HEROI + 1], 8, axis=0)
for nome in ("espelho", "desloca"):
    variantes = keras.ops.convert_to_numpy(cnn_aum.get_layer(nome)(variantes, training=True))
grade(np.clip(variantes, 0, 1), [f"sorteio {i + 1}" for i in range(8)])

O `EarlyStopping` interrompe o treino quando a perda na validação passa 5
épocas sem melhorar e devolve os pesos da melhor época (`restore_best_weights=True`); o
`ModelCheckpoint` grava essa época em `saida/melhor.keras`. Os dois olham a validação,
nunca o teste: se o teste escolhesse a época, ele deixaria de medir o desempenho em
fotos novas.

In [ ]:
med_com = AcuraciaDeTreino(x_peq, y_peq)
parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                      restore_best_weights=True, verbose=1)
guardar = keras.callbacks.ModelCheckpoint(SAIDA / "melhor.keras", monitor="val_loss",
                                          save_best_only=True)
h_com = treinar(cnn_aum, x_peq, y_peq, x_val, y_val, epocas=EPOCAS_SOBREAJUSTE,
                rotulo="treino com aumento", callbacks=[med_com, parar, guardar])
print("melhor época (menor perda na validação):", parar.best_epoch + 1)
if parar.stopped_epoch:
    print("o EarlyStopping parou o treino na época", parar.stopped_epoch + 1)

## As curvas

Traço contínuo: treino, medido sem aumento pelo *callback*. Tracejado: validação. Em
cinza, as curvas do professor (o vídeo do sobreajuste, médias entre sementes), quando a
análise delas já existe.

In [ ]:
def onde_pararia(perdas, paciencia=5):
    """A regra do EarlyStopping aplicada a uma lista de perdas de validação (épocas 1, 2, ...)."""
    melhor, espera, melhor_ep = np.inf, 0, 0
    for e, v in enumerate(perdas, start=1):
        if v < melhor:
            melhor, espera, melhor_ep = v, 0, e
        else:
            espera += 1
            if espera >= paciencia:
                return e, melhor_ep
    return None, melhor_ep


e4 = REF_ANALISE["e4"]
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (titulo, med, h, chave) in zip(eixos, (("sem aumento", med_sem, h_sem, "sem"),
                                               ("com aumento", med_com, h_com, "com"))):
    ep = np.arange(1, len(med.acc) + 1)
    ax.plot(ep, 100 * np.array(med.acc), "-", color="k", label="treino, aqui")
    ax.plot(ep, 100 * np.array(h.history["val_accuracy"]), "--", color="k", label="validação, aqui")
    if e4 is not None:
        ep4 = np.arange(len(e4[chave]["val_acc"]))
        ax.plot(ep4, 100 * np.array(e4[chave]["treino_acc"]), "-", color="0.65", label="treino, professor")
        ax.plot(ep4, 100 * np.array(e4[chave]["val_acc"]), "--", color="0.65", label="validação, professor")
    ax.set_title(titulo, fontsize=9); ax.set_xlabel("época"); ax.legend(fontsize=7)
eixos[0].set_ylabel("acurácia (%)")
plt.tight_layout(); plt.show()

parada, melhor_ep = onde_pararia(h_sem.history["val_loss"])
print(f"sem aumento: menor perda na validação na época {melhor_ep}; "
      + (f"o EarlyStopping teria parado na época {parada}" if parada else "o EarlyStopping não teria parado"))
if e4 is None:
    print("curvas do professor: a medir")
else:
    for chave in ("sem", "com"):
        paradas = [p["parada"] for p in e4[chave]["early_stopping"]]
        print(f"professor, {chave} aumento, {e4['epocas']} épocas: o EarlyStopping teria parado nas épocas {paradas}")

## O teste, uma vez

Só agora, com as escolhas feitas pela validação, as redes desta seção são medidas no
teste.

In [ ]:
melhor_sem = keras.models.load_model(SAIDA / "melhor_sem_aumento.keras")
for rotulo, modelo in (("sem aumento, última época", cnn_sem),
                       ("sem aumento, melhor época (ModelCheckpoint)", melhor_sem),
                       ("com aumento, melhor época (EarlyStopping)", cnn_aum)):
    print(f"{rotulo:45s}{pct(acuracia_teste(modelo, x_te)):>8s}")
if e4 is not None:
    for chave in ("sem", "com"):
        print(f"professor, {chave} aumento, última de {e4['epocas']} épocas, média de "
              f"{e4['sementes']} sementes: {pct(np.mean(e4[chave]['teste_acc']))}")

# 12 — Exercícios

Cada exercício treina uma ou duas redes; a seção 0 estima quanto cada treino leva nesta
máquina. As variações da CNN partem sempre da rede da aula: a função `variante` recria
as camadas de `construir_cnn()` a partir da configuração delas e troca só a peça pedida.
A função `quadrado_central` serve ao exercício 8 e ao apêndice.

In [ ]:
from PIL import Image


def variante(troca, aumento=False, nome="variante"):
    """A CNN da aula com peças trocadas. `troca(camada)` devolve a camada nova, ou None
    para manter a da aula. Todas as camadas são recriadas, com pesos novos sorteados."""
    novas = [keras.Input((LADO, LADO, 3))]
    for c in construir_cnn(aumento=aumento).layers:
        nova = troca(c)
        novas.append(type(c).from_config(c.get_config()) if nova is None else nova)
    return keras.Sequential(novas, name=nome)


def quadrado_central(caminho, lado):
    """Abre a foto, recorta o quadrado central e reduz a lado × lado px (0 a 1, RGB)."""
    im = Image.open(caminho).convert("RGB")
    m = min(im.size)
    x0, y0 = (im.width - m) // 2, (im.height - m) // 2
    im = im.crop((x0, y0, x0 + m, y0 + m)).resize((lado, lado), Image.Resampling.BOX)
    return np.asarray(im, np.float32) / 255


assert variante(lambda c: None).count_params() == PARAMS_CNN      # sem troca, é a rede da aula

### Exercício 1 — Embaralhar só dentro de blocos

`permutacao_blocos(b)` troca os pixels só dentro de cada bloco de b × b px; com b = 32,
é a permutação da aula. Treine a CNN com b = 4 e b = 8 px e compare com as fotos
(b = 1) e com as embaralhadas (b = 32). Quanto de vizinhança a CNN usa? Relacione com o
campo receptivo da seção 8.

In [ ]:
grade([embaralhar(x_te[HEROI:HEROI + 1], permutacao_blocos(b))[0] for b in (1, 2, 4, 8, 16, 32)],
      [f"b = {b} px" for b in (1, 2, 4, 8, 16, 32)], lado=1.6)

acc_blocos = {}
for b in (4, 8):
    perm_b = permutacao_blocos(b)
    # embaralhe treino, validação e teste com perm_b, treine uma CNN nova com treinar()
    # e guarde em acc_blocos[b] a acurácia no teste (acuracia_teste)
    ...

### Exercício 2 — Kernel 5 × 5 na primeira camada

Quantos parâmetros a conv1 passa a ter? Faça a conta pela fórmula da seção 7 antes de
conferir com `count_params()`. O campo receptivo muda (use `campo_receptivo`)? E a
acurácia nas fotos?

In [ ]:
def conv1_5x5(c):
    if c.name == "conv1":
        return layers.Conv2D(c.filters, 5, padding="same", name="conv1")


cnn_5x5 = compilar(variante(conv1_5x5))
print("parâmetros da conv1:", cnn_5x5.get_layer("conv1").count_params())
print(campo_receptivo(cnn_5x5)[0])
# treine com treinar() e meça com acuracia_teste()
...

### Exercício 3 — Média no lugar do máximo

Troque os três `MaxPooling2D` por `AveragePooling2D` do mesmo tamanho. O número de
parâmetros muda? O que muda nos mapas do navio (seção 8)? E a acurácia nas fotos?

In [ ]:
def media_no_lugar_do_maximo(c):
    if isinstance(c, layers.MaxPooling2D):
        return layers.AveragePooling2D(c.pool_size, name=c.name)


cnn_media = compilar(variante(media_no_lugar_do_maximo))
...

### Exercício 4 — Passo 2 no lugar do *pooling*

Em vez de reduzir o mapa com o *max pooling*, cada convolução anda de 2 em 2 px
(`strides=2`), e os *poolings* saem. Confira as formas do `summary()` com a fórmula da
seção 6. O que acontece com o tempo de cada época e com a acurácia?

In [ ]:
def passo_2(c):
    if isinstance(c, layers.Conv2D):
        return layers.Conv2D(c.filters, c.kernel_size, strides=2, padding="same", name=c.name)
    if isinstance(c, layers.MaxPooling2D):
        return layers.Identity(name=c.name)


cnn_passo = compilar(variante(passo_2))
cnn_passo.summary()
...

### Exercício 5 — Média global no lugar da fila

`GlobalAveragePooling2D` resume cada mapa da última camada num único número, a média
dele. Quantos parâmetros a camada densa perde? Treine nas fotos e nas embaralhadas. A
média global ignora a posição de cada valor: o que isso deveria mudar nas embaralhadas?

In [ ]:
def media_global(c):
    if isinstance(c, layers.Flatten):
        return layers.GlobalAveragePooling2D(name=c.name)


cnn_gap = compilar(variante(media_global))
print(f"parâmetros: {br(cnn_gap.count_params(), 0)} (a CNN da aula tem {br(PARAMS_CNN, 0)})")
...

### Exercício 6 — O ruído de semente

Treine a CNN nas fotos mais duas vezes, chamando `keras.utils.set_random_seed(1)` e
depois `(2)` antes de construir a rede. Com as três acurácias (a da seção 7 e as duas
novas), calcule a média e o desvio-padrão (`np.std(..., ddof=1)`) e compare com o σ do
professor no mesmo modo.

In [ ]:
accs_sementes = [ACC["cnn_fotos"]]
for semente in (1, 2):
    keras.utils.set_random_seed(semente)
    # construa, treine e meça uma CNN nova; acrescente a acurácia em accs_sementes
    ...

### Exercício 7 — Outra permutação

A identidade da seção 4 vale para qualquer permutação. Sorteie outra
(`permutacao(semente=7)`), refaça a rota 1 com ela e confira que as previsões continuam
iguais. Por que nada na conta dependia da permutação escolhida?

In [ ]:
perm7 = permutacao(semente=7)
x_te_emb7 = embaralhar(x_te, perm7)
# monte o kernel permutado com perm7, carregue-o numa construir_mlp() nova (como na
# seção 4) e compare as previsões nas fotos embaralhadas com perm7
...

### Exercício 8 — Uma foto própria em 32 × 32 px

Ponha uma foto (JPEG ou PNG) na pasta do notebook com o nome `minha_foto.jpg`, ou troque
o nome abaixo. A célula recorta o quadrado central, reduz a 32 × 32 px e pede a classe à
CNN da seção 7. A foto precisa mostrar uma das 10 classes. O que se perde na redução? A
rede acerta? E se a mesma foto for embaralhada com `perm`?

In [ ]:
FOTO = Path("minha_foto.jpg")
if FOTO.exists():
    foto32 = quadrado_central(FOTO, LADO)
    p = cnn_fotos.predict(foto32[None], verbose=0)[0]
    fig, eixos = plt.subplots(1, 2, figsize=(7, 2.8), gridspec_kw={"width_ratios": [1, 2.2]})
    eixos[0].imshow(foto32, interpolation="nearest"); eixos[0].set_axis_off()
    eixos[0].set_title("32 × 32 px", fontsize=8)
    eixos[1].barh(CLASSES, 100 * p, color="0.6"); eixos[1].invert_yaxis()
    eixos[1].set_xlabel("probabilidade (%)")
    plt.tight_layout(); plt.show()
else:
    print(f"{FOTO} não está na pasta do notebook")

# 13 — Apêndice opcional: uma rede pré-treinada

*Opcional; de preferência no Colab.* A MobileNetV2 foi treinada no ImageNet: 1.000
classes, fotos de 224 × 224 px. Na primeira vez, o Keras baixa os pesos do
`storage.googleapis.com` (da ordem de 14 MB, numa velocidade que não foi medida na rede
da Ilum) e a lista de nomes das classes. Por isso a célula só roda com
`RODAR_APENDICE = True`. Ela usa a foto do exercício 8 e a função `quadrado_central` da
seção 12; nada daqui é usado no resto do notebook. Redes maiores, como a ResNet50 e a
VGG16, fazem o mesmo papel com arquivos de pesos muito maiores.

In [ ]:
RODAR_APENDICE = False       # True baixa os pesos da MobileNetV2 na primeira vez

if RODAR_APENDICE:
    from keras.applications import mobilenet_v2

    FOTO = Path("minha_foto.jpg")
    with cronometro("carregar a MobileNetV2 (com o download, na primeira vez)"):
        mobilenet = keras.applications.MobileNetV2(weights="imagenet")
    x224 = mobilenet_v2.preprocess_input(255 * quadrado_central(FOTO, 224)[None])
    for _, nome, p in mobilenet_v2.decode_predictions(mobilenet.predict(x224, verbose=0), top=5)[0]:
        print(f"{nome:25s}{pct(float(p)):>8s}")
else:
    print("apêndice desligado: RODAR_APENDICE = False")

## Para ir além

*Dropout* e normalização em lote (`BatchNormalization`) são as regularizações mais
comuns contra o sobreajuste. A **transferência de aprendizado** parte de uma rede
pré-treinada, como a do apêndice, e ajusta só as últimas camadas aos dados novos.
Nenhuma delas foi medida nesta aula.